# Overture Streets Network Pipeline

Single-notebook workflow: **Download → Topology Repair → Modal Filters → Cost Attributes → Boundary Attribution → Network Datasets**

| Phase | Steps | Output |
|---|---|---|
| 1 — Download | 1–5 | `Segments`, `Connectors`, `Edges_Routable` in GDB |
| 2 — Filter | 6–7 | `Overture_Walk_Routable`, `Overture_Bike_Routable`, `Overture_Drive_Routable` |
| 3 — Cost attributes | 8 | `WalkTime`, `BikeTime`, `DriveTime` fields on each FC |
| 4 — Boundary attribution | 9–10 | `In_Basin` on all FCs; `Wilderness_Closed` on bike FC |
| 5 — Network Datasets | 11 | `Overture_Walk_ND`, `Overture_Bike_ND`, `Overture_Drive_ND` |

### Prerequisites
| Requirement | Notes |
|---|---|
| ArcGIS Pro 3.x | With **Network Analyst** extension |
| ND template XMLs | Walk: from `Overture_Walk_Network_ND.ipynb` Step 5; Bike/Drive: create in Pro UI (see Step 11) |
| Internet access | Overture S3 + TRPA map services |

In [16]:
import os

# =============================================================================
# CONFIGURATION — edit this section; all other cells run unchanged
# =============================================================================

# Overture release — check https://docs.overturemaps.org/release/ for latest
OVERTURE_RELEASE = "2026-05-20.0"

# Bounding box (WGS84 decimal degrees) — Tahoe Basin + Truckee + Carson City
BBOX = {"xmin": -120.40, "ymin": 38.55, "xmax": -119.65, "ymax": 39.50}

# File Geodatabase
GDB_PATH = r"F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb"
FDS_NAME = "OvertureStreets"

# Network Dataset template XMLs — keep in the same folder as this notebook
# Walk:  produced by Overture_Walk_Network_ND.ipynb Step 5
# Bike / Drive: create in ArcGIS Pro UI, then right-click ND > Export Network Dataset Template
TEMPLATE_DIR = os.getcwd()

# ---------------------------------------------------------------------------
# DERIVED PATHS
# ---------------------------------------------------------------------------
FDS_PATH = os.path.join(GDB_PATH, FDS_NAME)

print(f"GDB:     {GDB_PATH}")
print(f"FDS:     {FDS_PATH}")
print(f"Release: {OVERTURE_RELEASE}")
print(f"BBox:    {BBOX}")

GDB:     F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb
FDS:     F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets
Release: 2026-05-20.0
BBox:    {'xmin': -120.4, 'ymin': 38.55, 'xmax': -119.65, 'ymax': 39.5}


---
## Phase 1 — Download
Steps 1–5 download Overture `segment` and `connector` records, repair topology by splitting segments
at interior connector positions, and write three raw feature classes to the GDB.

### Step 1: Environment Setup

In [17]:
%pip install overturemaps geopandas pyarrow shapely --quiet

import os, sys, warnings
import pyproj

# ArcGIS Pro conda envs don't expose PROJ automatically — fix before any CRS use
_proj_data = r"C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Library\share\proj"
pyproj.datadir.set_data_dir(_proj_data)
try:
    _ = pyproj.CRS("EPSG:4326")
    print(f"PROJ OK — {pyproj.datadir.get_data_dir()}")
except Exception as e:
    raise RuntimeError(f"PROJ broken: {e}")

import pyarrow as pa
import overturemaps
import geopandas as gpd
import pandas as pd
from shapely import from_wkb
from shapely.ops import substring
from shapely.geometry import LineString, MultiLineString
warnings.filterwarnings("ignore")

print(f"geopandas {gpd.__version__} | pyarrow {pa.__version__}")

Note: you may need to restart the kernel to use updated packages.
PROJ OK — C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Library\share\proj
geopandas 1.1.3 | pyarrow 23.0.0


### Step 2: Download Segments and Connectors

In [18]:
bbox = (BBOX["xmin"], BBOX["ymin"], BBOX["xmax"], BBOX["ymax"])

print("Downloading segments (overturemaps auto-selects latest release)...")
seg_table = overturemaps.record_batch_reader("segment",   bbox=bbox).read_all()
print(f"  {seg_table.num_rows:,} segment records")

print("Downloading connectors...")
con_table = overturemaps.record_batch_reader("connector", bbox=bbox).read_all()
print(f"  {con_table.num_rows:,} connector records")

  74,948 segment records
  109,079 connector records


### Step 3: Flatten and Extract Road Attributes

Overture's raw PyArrow table contains nested struct/list columns (names, road_surface,
speed_limits). Flatten the table to drop struct columns that pandas can't hold natively,
then pull the fields we need from the raw table before flattening.

In [19]:
def table_to_gdf(table, geom_col="geometry", crs="EPSG:4326"):
    df = table.to_pandas()
    df["geometry"] = df[geom_col].apply(from_wkb)
    return gpd.GeoDataFrame(df, geometry="geometry", crs=crs)

def flatten_table(table):
    keep = [
        f.name for f in table.schema
        if not (pa.types.is_struct(f.type)
                or pa.types.is_large_list(f.type)
                or pa.types.is_list(f.type))
    ]
    return table.select(keep)

# Segments
segments_gdf = table_to_gdf(flatten_table(seg_table))
if "subtype" in segments_gdf.columns:
    segments_gdf = segments_gdf[segments_gdf["subtype"] == "road"].copy()

# Extract nested fields from raw table before it's discarded
seg_raw = seg_table.to_pandas()
if "names" in seg_raw.columns:
    segments_gdf["name"] = seg_raw["names"].apply(
        lambda x: x.get("primary") if isinstance(x, dict) else None
    )
if "road_surface" in seg_raw.columns:
    segments_gdf["surface"] = seg_raw["road_surface"].apply(
        lambda x: x[0].get("value") if isinstance(x, list) and x else None
    )
if "speed_limits" in seg_raw.columns:
    def _get_speed(x):
        if isinstance(x, list) and x:
            s = x[0].get("max_speed")
            return s.get("value") if isinstance(s, dict) else None
    segments_gdf["speed_limit_kmh"] = seg_raw["speed_limits"].apply(_get_speed)

# Connectors
connectors_gdf = table_to_gdf(flatten_table(con_table))

print(f"Road segments: {len(segments_gdf):,}")
print(f"Connectors:    {len(connectors_gdf):,}")
print(f"\nRoad class breakdown:")
print(segments_gdf["class"].value_counts().to_string())

Road segments: 74,867
Connectors:    109,079

Road class breakdown:
class
service          26054
residential      17665
footway           9534
track             8109
path              4806
tertiary          2592
secondary         1606
primary           1237
unclassified      1190
unknown            701
trunk              506
motorway           344
cycleway           275
steps              186
living_street       38
pedestrian          16
bridleway            8


### Step 4: Topology Repair — Split and Snap

Overture segments declare interior connector positions via `at` values (0–1 fractional position
along the LineString). Two segments that share a connector but whose geometries only overlap
at a mid-point won't connect in a Network Dataset unless the segment is split there first.

**Pipeline:**
1. Extract `{connector_id, at}` refs from the raw `connectors` column
2. Split each segment at every interior connector position (producing sub-edges)
3. Snap each sub-edge endpoint to the connector's exact point coordinate
4. Join road attributes and compute geodesic length in metres

In [20]:
# ---- 4a. Extract connector attachment refs --------------------------------
def extract_connector_refs(table):
    rows = []
    for seg_id, clist in zip(table["id"].to_pylist(), table["connectors"].to_pylist()):
        if not clist:
            continue
        for c in clist:
            if not isinstance(c, dict):
                continue
            cid = c.get("connector_id") or c.get("id")
            at  = c.get("at")
            if cid is not None and at is not None:
                rows.append({"segment_id": seg_id, "connector_id": cid, "at": float(at)})
    return pd.DataFrame(rows, columns=["segment_id", "connector_id", "at"])

conn_refs = extract_connector_refs(seg_table)
interior  = conn_refs[(conn_refs["at"] > 1e-8) & (conn_refs["at"] < 1 - 1e-8)]
print(f"Attachment records:           {len(conn_refs):,}")
print(f"Interior splits required:     {len(interior):,}")
print(f"Segments with at least 1 ref: {conn_refs['segment_id'].nunique():,}")

Attachment records:           222,525
Interior splits required:     72,626
Segments with at least 1 ref: 74,948


In [21]:
# ---- 4b. Split + snap -------------------------------------------------------
def split_at_positions(geom, at_values):
    if isinstance(geom, MultiLineString):
        geom = list(geom.geoms)[0]
    cuts = sorted({0.0} | {float(v) for v in at_values} | {1.0})
    pieces = []
    for i in range(len(cuts) - 1):
        a, b = cuts[i], cuts[i + 1]
        if b - a < 1e-9:
            continue
        sub = substring(geom, a, b, normalized=True)
        if sub is not None and not sub.is_empty and len(sub.coords) >= 2:
            pieces.append((sub, a, b))
    return pieces or [(geom, 0.0, 1.0)]

refs_grouped = conn_refs.groupby("segment_id")
seg_index    = segments_gdf.set_index("id")
edge_rows    = []

for seg_id, group in refs_grouped:
    if seg_id not in seg_index.index:
        continue
    geom      = seg_index.at[seg_id, "geometry"]
    at_vals   = group["at"].tolist()
    at_to_con = {round(float(r.at), 10): r.connector_id for r in group.itertuples()}
    for sub_geom, from_at, to_at in split_at_positions(geom, at_vals):
        edge_rows.append({
            "segment_id":     seg_id,
            "from_connector": at_to_con.get(round(from_at, 10)),
            "to_connector":   at_to_con.get(round(to_at,   10)),
            "geometry":       sub_geom,
        })

# Segments with no connector refs kept as-is (dead ends, isolated paths)
segs_with_refs = set(refs_grouped.groups) & set(seg_index.index)
for seg_id in set(seg_index.index) - segs_with_refs:
    edge_rows.append({
        "segment_id":     seg_id,
        "from_connector": None,
        "to_connector":   None,
        "geometry":       seg_index.at[seg_id, "geometry"],
    })

edges_gdf = gpd.GeoDataFrame(edge_rows, geometry="geometry", crs=segments_gdf.crs)

# Snap endpoints to exact connector coordinates
con_point = connectors_gdf.set_index("id")["geometry"].to_dict()

def snap_endpoints(geom, from_con, to_con, lookup):
    coords = list(geom.coords)
    if from_con and from_con in lookup:
        pt = lookup[from_con]; coords[0] = (pt.x, pt.y)
    if to_con and to_con in lookup:
        pt = lookup[to_con];   coords[-1] = (pt.x, pt.y)
    return LineString(coords) if len(coords) >= 2 else geom

edges_gdf["geometry"] = edges_gdf.apply(
    lambda r: snap_endpoints(r.geometry, r.from_connector, r.to_connector, con_point), axis=1
)
edges_gdf = edges_gdf.set_geometry("geometry")

# Join road class and attributes from segments
attr_cols = [c for c in ["class", "name", "surface", "speed_limit_kmh"] if c in segments_gdf.columns]
edges_gdf = edges_gdf.merge(
    segments_gdf[["id"] + attr_cols].rename(columns={"id": "segment_id"}),
    on="segment_id", how="left",
)

# Geodesic length in metres
edges_gdf["length_m"] = edges_gdf.to_crs("EPSG:32610").geometry.length.round(2)

print(f"Raw segments:   {len(segments_gdf):,}")
print(f"Edges (split):  {len(edges_gdf):,}  (+{len(edges_gdf) - len(segments_gdf):,} from interior splits)")
print(f"Columns: {list(edges_gdf.columns)}")

Raw segments:   74,867
Edges (split):  147,359  (+72,492 from interior splits)
Columns: ['segment_id', 'from_connector', 'to_connector', 'geometry', 'class', 'name', 'surface', 'speed_limit_kmh', 'length_m']


### Step 5: Export to File Geodatabase

Writes three feature classes to `OvertureStreets`:
- `Overture_Segments` — raw Overture road segments (one row per original segment)
- `Overture_Connectors` — Overture intersection nodes
- `Overture_Edges_Routable` — split + snapped edges (input for the modal filter in Phase 2)

> **Note:** ArcGIS requires feature class names to be unique across the entire GDB, including
> across feature datasets. Generic names like `Segments` collide with existing FCs elsewhere
> in `Streets_Network.gdb`, so all Overture FCs are prefixed with `Overture_`.

In [22]:
import arcpy

def _create_fc_schema(gdf, fds_path, fc_name):
    """Delete/recreate FC and add attribute fields. Returns (mapped_cols, cursor_fields, is_float, is_int).
    Must be called OUTSIDE any edit session — schema changes cannot run inside one."""
    fc_path    = os.path.join(fds_path, fc_name)
    sr         = arcpy.SpatialReference(4326)
    geom_map   = {
        "Point":           "POINT",
        "MultiPoint":      "MULTIPOINT",
        "LineString":      "POLYLINE",
        "MultiLineString": "POLYLINE",
        "Polygon":         "POLYGON",
        "MultiPolygon":    "POLYGON",
    }
    arcpy_type = geom_map.get(gdf.geometry.geom_type.dropna().iloc[0], "POLYLINE")

    if arcpy.Exists(fc_path):
        try:
            arcpy.management.Delete(fc_path)
        except Exception as e:
            raise RuntimeError(
                f"Cannot delete {fc_name} — ArcGIS Pro has a schema lock.\n"
                f"Remove it from the map (right-click the layer → Remove), then rerun."
            ) from e
    arcpy.management.CreateFeatureclass(fds_path, fc_name, arcpy_type, spatial_reference=sr)
    if not arcpy.Exists(fc_path):
        raise RuntimeError(f"CreateFeatureclass silently failed — {fc_path} does not exist")

    reserved  = {f.name.lower() for f in arcpy.ListFields(fc_path)}
    attr_cols = [c for c in gdf.columns if c != "geometry"]
    field_map, used = {}, set(reserved)
    for col in attr_cols:
        base = col
        if base.lower() in used:
            for i in range(1, 100):
                candidate = f"{col}_{i}"
                if candidate.lower() not in used:
                    base = candidate; break
            else:
                print(f"  [{fc_name}] Skipping '{col}' — no unique field name found")
                continue
        field_map[col] = base
        used.add(base.lower())

    for col, fname in field_map.items():
        if pd.api.types.is_float_dtype(gdf[col]):
            arcpy.management.AddField(fc_path, fname, "DOUBLE", field_is_nullable="NULLABLE")
        elif pd.api.types.is_integer_dtype(gdf[col]):
            arcpy.management.AddField(fc_path, fname, "LONG",   field_is_nullable="NULLABLE")
        else:
            arcpy.management.AddField(fc_path, fname, "TEXT",   field_length=254, field_is_nullable="NULLABLE")

    renamed = {k: v for k, v in field_map.items() if k != v}
    if renamed:
        print(f"  [{fc_name}] Renamed fields: {renamed}")

    mapped_cols   = list(field_map.keys())
    cursor_fields = [field_map[c] for c in mapped_cols] + ["SHAPE@WKT"]
    is_float      = {c: pd.api.types.is_float_dtype(gdf[c])   for c in mapped_cols}
    is_int        = {c: pd.api.types.is_integer_dtype(gdf[c]) for c in mapped_cols}
    return mapped_cols, cursor_fields, is_float, is_int


def _insert_fc_data(gdf, fds_path, fc_name, mapped_cols, cursor_fields, is_float, is_int):
    """Insert rows via InsertCursor. Must be called inside an active edit session."""
    fc_path = os.path.join(fds_path, fc_name)
    with arcpy.da.InsertCursor(fc_path, cursor_fields) as cur:
        for _, row in gdf.iterrows():
            vals = []
            for col in mapped_cols:
                v = row[col]
                try:
                    null = pd.isna(v)
                except (TypeError, ValueError):
                    null = False
                if null:
                    vals.append(None if (is_float[col] or is_int[col]) else "")
                elif is_float[col] or is_int[col]:
                    vals.append(v)
                else:
                    vals.append(str(v)[:254])
            vals.append(row.geometry.wkt if row.geometry is not None else None)
            cur.insertRow(vals)
    n = int(arcpy.management.GetCount(fc_path).getOutput(0))
    print(f"  {fc_name:<26} {n:>8,} features  →  {fc_path}")


# Create feature dataset if needed
if not arcpy.Exists(FDS_PATH):
    arcpy.management.CreateFeatureDataset(GDB_PATH, FDS_NAME, arcpy.SpatialReference(4326))
    print(f"Created: {FDS_PATH}")

# Pre-flight: delete all NDs so source FCs can be safely deleted/recreated.
if arcpy.Exists(FDS_PATH):
    arcpy.env.workspace = FDS_PATH
    existing_nds = arcpy.ListDatasets("*", "Network") or []
    arcpy.env.workspace = None
    for nd_name in existing_nds:
        nd_p = os.path.join(FDS_PATH, nd_name)
        try:
            arcpy.management.Delete(nd_p)
            print(f"Removed ND: {nd_name}")
        except Exception as e:
            if arcpy.Exists(nd_p):
                raise RuntimeError(
                    f"Cannot delete {nd_name} — ArcGIS Pro has a schema lock.\n"
                    f"Remove it from the map (right-click → Remove), then rerun."
                ) from e
    if existing_nds:
        arcpy.management.Compact(GDB_PATH)
        print("GDB compacted.\n")

EXPORTS = [
    ("Overture_Segments",       segments_gdf),
    ("Overture_Connectors",     connectors_gdf),
    ("Overture_Edges_Routable", edges_gdf),
]

# Phase 1: create all FC schemas outside any edit session.
# Schema operations (CreateFeatureclass, AddField) cannot run inside an edit session,
# and interleaving them with InsertCursor edit sessions causes silent failures.
print("Creating schemas...")
schemas = {fc: _create_fc_schema(gdf, FDS_PATH, fc) for fc, gdf in EXPORTS}

# Phase 2: insert all data in a single edit session.
print("Inserting data...")
editor = arcpy.da.Editor(GDB_PATH)
editor.startEditing(False, True)
editor.startOperation()
try:
    for fc_name, gdf in EXPORTS:
        _insert_fc_data(gdf, FDS_PATH, fc_name, *schemas[fc_name])
    editor.stopOperation()
    editor.stopEditing(True)
except Exception:
    editor.abortOperation()
    editor.stopEditing(False)
    raise

print("Export complete.")
if existing_nds:
    print("\nNDs were removed — re-run Steps 7–11 to rebuild the modal FCs and NDs.")

Removed ND: Overture_Walk_ND
Removed ND: Overture_Bike_ND
Removed ND: Overture_Drive_ND
GDB compacted.

Creating schemas...
Inserting data...
  Overture_Segments            74,867 features  →  F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Segments
  Overture_Connectors         109,079 features  →  F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Connectors
  Overture_Edges_Routable     147,359 features  →  F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Edges_Routable
Export complete.

NDs were removed — re-run Steps 7–11 to rebuild the modal FCs and NDs.


---
## Phase 2 — Filter
Steps 6–7 verify the arcpy environment and apply class whitelists to `Edges_Routable`,
producing one feature class per travel mode.

### Step 6: Verify arcpy and Network Analyst

In [23]:
import arcpy, sys

print(f"Python: {sys.version}")
print(f"arcpy:  {arcpy.GetInstallInfo()['Version']}")

na_status = arcpy.CheckExtension("network")
if na_status == "Available":
    arcpy.CheckOutExtension("network")
    print("Network Analyst: checked out")
else:
    print(f"WARNING: Network Analyst status: {na_status}")

Python: 3.13.13 | packaged by Anaconda, Inc. | (main, Apr 22 2026, 16:49:16) [MSC v.1938 64 bit (AMD64)]
arcpy:  3.7
Network Analyst: checked out


### Step 7: Filter to Modal Feature Classes

`make_filtered_fc(fc_name, class_whitelist)` applies a SQL whitelist to `Overture_Edges_Routable`
and writes the result to `fc_name` in the same feature dataset, using the temp-name
pattern to avoid GDB Error 002851 on immediate re-creation.

Existing Network Datasets are deleted first to release any schema locks on the edge FCs.

| Mode | Excluded from `Overture_Edges_Routable` |
|---|---|
| Walk | motorway, trunk, motorway_link, trunk_link |
| Bike | motorway, trunk, motorway_link, trunk_link, steps, pedestrian |
| Drive | footway, pedestrian, path, cycleway, track, bridleway, steps |

In [24]:
import arcpy, os

# Delete all NDs in the FDS to release schema locks before touching edge FCs.
arcpy.env.workspace = FDS_PATH
existing_nds = arcpy.ListDatasets("*", "Network") or []
arcpy.env.workspace = None

for nd_name in existing_nds:
    nd_p = os.path.join(FDS_PATH, nd_name)
    try:
        arcpy.management.Delete(nd_p)
        print(f"Deleted: {nd_name}")
    except Exception as e:
        if arcpy.Exists(nd_p):
            raise RuntimeError(
                f"Cannot delete {nd_name} — ArcGIS Pro has a schema lock.\n"
                f"Remove it from the map (right-click → Remove), then rerun."
            ) from e

if existing_nds:
    arcpy.management.Compact(GDB_PATH)
    print("GDB compacted.\n")

FC_EDGES = os.path.join(FDS_PATH, "Overture_Edges_Routable")

def make_filtered_fc(fc_name, class_whitelist):
    out_path = os.path.join(FDS_PATH, fc_name)
    # Delete the old FC first so FeatureClassToFeatureClass can write to the same name.
    # The original temp-name+Copy pattern caused ERROR 000196 because the freshly written
    # temp FC was inaccessible to Copy before the GDB state settled.
    if arcpy.Exists(out_path):
        arcpy.management.Delete(out_path)
    total = int(arcpy.management.GetCount(FC_EDGES)[0])
    sql   = '"class" IN ({})'.format(", ".join(f"'{c}'" for c in sorted(class_whitelist)))
    arcpy.conversion.FeatureClassToFeatureClass(FC_EDGES, FDS_PATH, fc_name, sql)
    n_flt = int(arcpy.management.GetCount(out_path)[0])
    print(f"[{fc_name}] {total:,} → {n_flt:,} after whitelist (removed {total - n_flt:,})")
    return out_path

WALK_CLASSES = {
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "residential", "service",
    "living_street", "unclassified", "unknown",
    "footway", "pedestrian", "path", "cycleway", "track", "bridleway", "steps",
}
BIKE_CLASSES = {
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "residential", "service",
    "living_street", "unclassified", "unknown",
    "footway", "path", "cycleway", "track", "bridleway",
}
DRIVE_CLASSES = {
    "motorway", "motorway_link", "trunk", "trunk_link",
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "residential", "service",
    "living_street", "unclassified", "unknown",
}

make_filtered_fc("Overture_Walk_Routable",  WALK_CLASSES)
arcpy.management.Compact(GDB_PATH)
make_filtered_fc("Overture_Bike_Routable",  BIKE_CLASSES)
arcpy.management.Compact(GDB_PATH)
make_filtered_fc("Overture_Drive_Routable", DRIVE_CLASSES)

[Overture_Walk_Routable] 147,359 → 145,810 after whitelist (removed 1,549)
[Overture_Bike_Routable] 147,359 → 145,590 after whitelist (removed 1,769)
[Overture_Drive_Routable] 147,359 → 95,285 after whitelist (removed 52,074)


'F:\\GIS\\PROJECTS\\Transportation\\Streets Network\\Streets_Network.gdb\\OvertureStreets\\Overture_Drive_Routable'

---
## Phase 3 — Cost Attributes
Step 8 adds a travel-time field to each modal FC. `length_m` (calculated in Step 4 and
preserved by `FeatureClassToFeatureClass`) is used directly — no re-projection needed.
Drive speed defaults are class-based; the `speed_limit_kmh` field is used where populated.

### Step 8: Calculate Travel Time Fields

In [25]:
import arcpy, os

WALK_SPEED = {
    "primary": 5.0,  "primary_link": 5.0,  "secondary": 5.0,  "secondary_link": 5.0,
    "tertiary": 5.0, "tertiary_link": 5.0, "residential": 5.0, "service": 5.0,
    "living_street": 5.0, "unclassified": 5.0, "unknown": 5.0,
    "footway": 5.0, "pedestrian": 5.0, "path": 5.0,
    "cycleway": 4.5, "track": 4.0, "bridleway": 4.0, "steps": 3.0,
}
BIKE_SPEED = {
    "primary": 18.0, "primary_link": 18.0, "secondary": 18.0, "secondary_link": 18.0,
    "tertiary": 16.0, "tertiary_link": 16.0, "residential": 15.0, "service": 12.0,
    "living_street": 12.0, "unclassified": 13.0, "unknown": 13.0,
    "footway": 10.0, "path": 10.0, "cycleway": 18.0, "track": 8.0, "bridleway": 7.0,
}
DRIVE_SPEED = {
    "motorway": 100.0, "motorway_link": 70.0, "trunk": 80.0,  "trunk_link": 60.0,
    "primary":   60.0, "primary_link":  50.0, "secondary": 55.0, "secondary_link": 45.0,
    "tertiary":  50.0, "tertiary_link": 40.0, "residential": 40.0, "service": 25.0,
    "living_street": 20.0, "unclassified": 40.0, "unknown": 50.0,
}

COST_MODES = [
    ("Walk",  "Overture_Walk_Routable",  "WalkTime",  WALK_SPEED,  5.0),
    ("Bike",  "Overture_Bike_Routable",  "BikeTime",  BIKE_SPEED,  12.0),
    ("Drive", "Overture_Drive_Routable", "DriveTime", DRIVE_SPEED, 50.0),
]

# UpdateCursor on a ND source FC triggers dirty-area tracking, which fails if the ND
# is in an inconsistent state. Delete all NDs in the FDS first so the FCs are bare,
# then compact to flush dirty-area metadata. Re-run Step 11 after this cell.
arcpy.env.workspace = FDS_PATH
existing_nds = arcpy.ListDatasets("*", "Network") or []
arcpy.env.workspace = ""

for nd_name in existing_nds:
    nd_p = os.path.join(FDS_PATH, nd_name)
    try:
        arcpy.management.Delete(nd_p)
        print(f"Removed ND: {nd_name}")
    except Exception as e:
        if arcpy.Exists(nd_p):
            raise RuntimeError(
                f"Cannot delete {nd_name} — ArcGIS Pro has a schema lock.\n"
                f"Remove it from the map (right-click → Remove), then rerun."
            ) from e

if existing_nds:
    arcpy.management.Compact(GDB_PATH)
    print("GDB compacted.\n")

for mode, fc_name, time_field, speed_table, default_kmh in COST_MODES:
    fc_path = os.path.join(FDS_PATH, fc_name)
    if not arcpy.Exists(fc_path):
        print(f"[{mode}] SKIPPED — {fc_name} not found"); continue

    existing = {f.name for f in arcpy.ListFields(fc_path)}
    if time_field not in existing:
        arcpy.management.AddField(fc_path, time_field, "DOUBLE", field_is_nullable="NULLABLE")
        existing.add(time_field)

    has_len     = "length_m" in existing
    has_spd_lim = "speed_limit_kmh" in existing
    read_geom   = "length_m" if has_len else "SHAPE@"
    cursor_flds = ["class", read_geom]
    if mode == "Drive" and has_spd_lim:
        cursor_flds.append("speed_limit_kmh")
    cursor_flds.append(time_field)

    editor = arcpy.da.Editor(GDB_PATH)
    editor.startEditing(False, True)
    editor.startOperation()
    try:
        with arcpy.da.UpdateCursor(fc_path, cursor_flds) as cur:
            for row in cur:
                cls      = str(row[0]).lower().strip() if row[0] else "unknown"
                length_m = row[1] if has_len else (row[1].getLength("GEODESIC", "METERS") if row[1] else 0.0)
                if mode == "Drive" and has_spd_lim:
                    spd_lim = row[2]
                    speed   = float(spd_lim) if spd_lim and spd_lim > 0 else speed_table.get(cls, default_kmh)
                    row[-1] = (length_m / 1000.0) / speed * 60.0 if length_m else 0.0
                else:
                    speed   = speed_table.get(cls, default_kmh)
                    row[-1] = (length_m / 1000.0) / speed * 60.0 if length_m else 0.0
                cur.updateRow(row)
        editor.stopOperation()
        editor.stopEditing(True)
    except Exception:
        editor.abortOperation()
        editor.stopEditing(False)
        raise

    n = int(arcpy.management.GetCount(fc_path)[0])
    print(f"[{mode}] {time_field} calculated for {n:,} features")

if existing_nds:
    print("\nNDs were removed before this step — re-run Step 11 to rebuild them.")

[Walk] WalkTime calculated for 145,810 features
[Bike] BikeTime calculated for 145,590 features
[Drive] DriveTime calculated for 95,285 features


---
## Phase 4 — Boundary Attribution
Steps 9–10 download polygon layers from the TRPA Boundaries map service and attribute
each modal FC with `In_Basin` (all modes) and `Wilderness_Closed` (bike only).

`INTERSECT` is used for both: a segment crossing the boundary or touching a wilderness
area at any point is flagged, preserving cross-boundary routing connections.

### Step 9: TRPA Basin Boundary → `In_Basin`

In [26]:
import requests, json, tempfile, os, arcpy

BOUNDARY_REST  = "https://maps.trpa.org/server/rest/services/Boundaries/MapServer/4"
BASIN_FC_PATH  = os.path.join(GDB_PATH, "TRPA_Boundary")
IN_BASIN_FIELD = "In_Basin"

# Download boundary
meta = requests.get(BOUNDARY_REST, params={"f": "json"}).json()
print(f"Layer: {meta.get('name', '(unknown)')}")

resp = requests.get(BOUNDARY_REST + "/query", params={
    "where": "1=1", "outFields": "*", "returnGeometry": "true", "outSR": "4326", "f": "json"
})
resp.raise_for_status()
data = resp.json()
print(f"Features received: {len(data.get('features', []))}")

tmp_json = os.path.join(tempfile.gettempdir(), "TRPA_Boundary.json")
with open(tmp_json, "w") as f:
    json.dump(data, f)
if arcpy.Exists(BASIN_FC_PATH):
    arcpy.management.Delete(BASIN_FC_PATH)
arcpy.conversion.JSONToFeatures(tmp_json, BASIN_FC_PATH)
print(f"Imported → {BASIN_FC_PATH}")
print()

# Attribute all three modal FCs
for mode, fc_name in [("Walk",  "Overture_Walk_Routable"),
                      ("Bike",  "Overture_Bike_Routable"),
                      ("Drive", "Overture_Drive_Routable")]:
    fc_path = os.path.join(FDS_PATH, fc_name)
    if not arcpy.Exists(fc_path):
        print(f"[{mode}] SKIPPED — {fc_name} not found"); continue
    if IN_BASIN_FIELD not in {f.name for f in arcpy.ListFields(fc_path)}:
        arcpy.management.AddField(fc_path, IN_BASIN_FIELD, "SHORT", field_is_nullable="NULLABLE")
    arcpy.management.CalculateField(fc_path, IN_BASIN_FIELD, "0", "PYTHON3")
    lyr = f"{fc_name}_lyr"
    arcpy.management.MakeFeatureLayer(fc_path, lyr)
    arcpy.management.SelectLayerByLocation(lyr, "INTERSECT", BASIN_FC_PATH, selection_type="NEW_SELECTION")
    n_in = int(arcpy.management.GetCount(lyr)[0])
    if n_in > 0:
        arcpy.management.CalculateField(lyr, IN_BASIN_FIELD, "1", "PYTHON3")
    arcpy.management.Delete(lyr)
    total = int(arcpy.management.GetCount(fc_path)[0])
    print(f"[{mode:<5}] {n_in:>6,} in basin  |  {total - n_in:>6,} outside  |  {total:>6,} total")

Layer: TRPA Boundary
Features received: 1
Imported → F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\TRPA_Boundary

[Walk ] 24,948 in basin  |  120,862 outside  |  145,810 total
[Bike ] 24,865 in basin  |  120,725 outside  |  145,590 total
[Drive] 18,285 in basin  |  77,000 outside  |  95,285 total


### Step 10: Wilderness Areas → `Wilderness_Closed` (Bike FC only)

Any bike segment that intersects a designated wilderness area is flagged `Wilderness_Closed = 1`.
When building the Bike Network Dataset use the definition query:
```
In_Basin = 1 AND Wilderness_Closed = 0
```

In [27]:
import requests, json, tempfile, os, arcpy

WILDERNESS_REST  = "https://maps.trpa.org/server/rest/services/Boundaries/MapServer/13"
WILDERNESS_PATH  = os.path.join(GDB_PATH, "Wilderness_Areas")
BIKE_FC_PATH     = os.path.join(FDS_PATH, "Overture_Bike_Routable")
WILDERNESS_FIELD = "Wilderness_Closed"

# Download wilderness polygons
meta = requests.get(WILDERNESS_REST, params={"f": "json"}).json()
print(f"Layer: {meta.get('name', '(unknown)')}")

resp = requests.get(WILDERNESS_REST + "/query", params={
    "where": "1=1", "outFields": "*", "returnGeometry": "true", "outSR": "4326", "f": "json"
})
resp.raise_for_status()
data = resp.json()

n_wild = len(data.get("features", []))
print(f"Features received: {n_wild}")
for feat in data.get("features", []):
    attrs = feat.get("attributes", {})
    name  = next((v for k, v in attrs.items() if "name" in k.lower() and v), list(attrs.values())[1])
    print(f"  {name}")

tmp_json = os.path.join(tempfile.gettempdir(), "Wilderness_Areas.json")
with open(tmp_json, "w") as f:
    json.dump(data, f)
if arcpy.Exists(WILDERNESS_PATH):
    arcpy.management.Delete(WILDERNESS_PATH)
arcpy.conversion.JSONToFeatures(tmp_json, WILDERNESS_PATH)
print(f"Imported → {WILDERNESS_PATH}")
print()

# Attribute bike FC
if not arcpy.Exists(BIKE_FC_PATH):
    raise RuntimeError("Overture_Bike_Routable not found — run Step 7 first.")

if WILDERNESS_FIELD not in {f.name for f in arcpy.ListFields(BIKE_FC_PATH)}:
    arcpy.management.AddField(BIKE_FC_PATH, WILDERNESS_FIELD, "SHORT", field_is_nullable="NULLABLE")
arcpy.management.CalculateField(BIKE_FC_PATH, WILDERNESS_FIELD, "0", "PYTHON3")

lyr = "bike_wild_lyr"
arcpy.management.MakeFeatureLayer(BIKE_FC_PATH, lyr)
arcpy.management.SelectLayerByLocation(lyr, "INTERSECT", WILDERNESS_PATH, selection_type="NEW_SELECTION")
n_closed = int(arcpy.management.GetCount(lyr)[0])
if n_closed > 0:
    arcpy.management.CalculateField(lyr, WILDERNESS_FIELD, "1", "PYTHON3")
arcpy.management.Delete(lyr)

total = int(arcpy.management.GetCount(BIKE_FC_PATH)[0])
print(f"Wilderness_Closed = 1 (prohibited): {n_closed:>6,}")
print(f"Wilderness_Closed = 0 (open):        {total - n_closed:>6,}")
print(f"Total:                               {total:>6,}")

Layer: Wilderness
Features received: 3
  Desolation Wilderness
  Granite Chief Wilderness
  Mt. Rose Wilderness
Imported → F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\Wilderness_Areas

Wilderness_Closed = 1 (prohibited):    295
Wilderness_Closed = 0 (open):        145,295
Total:                               145,590


---
## Phase 5 — Network Datasets

Step 11a generates the Bike and Drive template XMLs automatically from the Walk template
(produced by `Overture_Walk_Network_ND.ipynb`). Step 11b creates and builds all three NDs.

### ArcGIS Pro 3.x — where travel attributes actually live

The **New Network Dataset** wizard only configures name and sources — travel attributes are
**not** in the wizard. They're added after creation through **Properties**:

1. Catalog pane → right-click `OvertureStreets` → **New Network Dataset**
2. Name it, click **Next** → add the edge source FC, connectivity: **Any Vertex** → **Finish**
3. Right-click the new ND → **Properties**
4. **Travel Attributes** tab → **+** → Name: `BikeTime`, Usage: Cost, Units: Minutes, Data Type: Double
5. Click the attribute row → **Evaluators** → set both directions: Type: **Field**, Expression: `!BikeTime!`
6. **Travel Modes** tab → **+** → Name: Biking, Impedance: BikeTime → **OK**
7. Right-click ND → **Build Network**
8. Right-click ND → **Export Network Dataset Template** → save as `overture_bike_nd_template.xml`

> Step 11a does this automatically for Bike and Drive by rewriting the Walk template XML —
> you only need the manual steps above if you want to customise travel modes beyond what's here.

### Step 11a: Generate Bike and Drive Templates from Walk Template

Reads `overture_walk_nd_template.xml` and does targeted substitutions to produce
`overture_bike_nd_template.xml` and `overture_drive_nd_template.xml`.

Substitutions per mode:

| What | Walk | Bike | Drive |
|---|---|---|---|
| Edge source FC | `Overture_Walk_Routable` | `Overture_Bike_Routable` | `Overture_Drive_Routable` |
| ND name | `Overture_Walk_ND` | `Overture_Bike_ND` | `Overture_Drive_ND` |
| Cost attribute | `WalkTime` | `BikeTime` | `DriveTime` |
| Travel mode name | `Walking` | `Biking` | `Driving` |

In [28]:
import os

WALK_TEMPLATE = os.path.join(TEMPLATE_DIR, "overture_walk_nd_template.xml")

DERIVED_MODES = [
    {
        "out":      os.path.join(TEMPLATE_DIR, "overture_bike_nd_template.xml"),
        "fc_from":  "Overture_Walk_Routable",  "fc_to":  "Overture_Bike_Routable",
        "nd_from":  "Overture_Walk_ND",        "nd_to":  "Overture_Bike_ND",
        "att_from": "WalkTime",                "att_to": "BikeTime",
        "mode_from":"Walking",                 "mode_to":"Biking",
    },
    {
        "out":      os.path.join(TEMPLATE_DIR, "overture_drive_nd_template.xml"),
        "fc_from":  "Overture_Walk_Routable",  "fc_to":  "Overture_Drive_Routable",
        "nd_from":  "Overture_Walk_ND",        "nd_to":  "Overture_Drive_ND",
        "att_from": "WalkTime",                "att_to": "DriveTime",
        "mode_from":"Walking",                 "mode_to":"Driving",
    },
]

if not os.path.exists(WALK_TEMPLATE):
    raise FileNotFoundError(
        f"Walk template not found: {WALK_TEMPLATE}\n"
        f"Run Overture_Walk_Network_ND.ipynb Step 5 first, or export it manually from ArcGIS Pro."
    )

with open(WALK_TEMPLATE, "r", encoding="utf-8") as f:
    walk_xml = f.read()

for m in DERIVED_MODES:
    xml = walk_xml
    for src, dst in [
        (m["fc_from"],   m["fc_to"]),
        (m["nd_from"],   m["nd_to"]),
        (m["att_from"],  m["att_to"]),
        (m["mode_from"], m["mode_to"]),
    ]:
        xml = xml.replace(src, dst)

    with open(m["out"], "w", encoding="utf-8") as f:
        f.write(xml)
    print(f"Generated: {os.path.basename(m['out'])}")

Generated: overture_bike_nd_template.xml
Generated: overture_drive_nd_template.xml


In [29]:
import arcpy, os, tempfile

ND_MODES = [
    {
        "mode":       "Walk",
        "fc_name":    "Overture_Walk_Routable",
        "nd_name":    "Overture_Walk_ND",
        "time_field": "WalkTime",
        "template":   os.path.join(TEMPLATE_DIR, "overture_walk_nd_template.xml"),
    },
    {
        "mode":       "Bike",
        "fc_name":    "Overture_Bike_Routable",
        "nd_name":    "Overture_Bike_ND",
        "time_field": "BikeTime",
        "template":   os.path.join(TEMPLATE_DIR, "overture_bike_nd_template.xml"),
    },
    {
        "mode":       "Drive",
        "fc_name":    "Overture_Drive_Routable",
        "nd_name":    "Overture_Drive_ND",
        "time_field": "DriveTime",
        "template":   os.path.join(TEMPLATE_DIR, "overture_drive_nd_template.xml"),
    },
]

STALE_SOURCE = "Overture_Walk_Segments"

# --- Pre-flight 1: verify time fields exist on each source FC ---
# CreateNetworkDatasetFromTemplate validates field evaluators before creating anything.
# If Step 8 hasn't run since the last Step 7, the time fields won't be present.
missing = []
for m in ND_MODES:
    fc_path = os.path.join(FDS_PATH, m["fc_name"])
    if not arcpy.Exists(fc_path):
        missing.append(f"  {m['fc_name']} — FC not found (run Step 7)")
    elif m["time_field"] not in {f.name for f in arcpy.ListFields(fc_path)}:
        missing.append(f"  {m['fc_name']} — missing field '{m['time_field']}' (run Step 8)")

if missing:
    raise RuntimeError("Cannot build NDs — prerequisites missing:\n" + "\n".join(missing))

print("Pre-flight: all source FCs and time fields present.\n")

# --- Pre-flight 2: delete every ND in the FDS, then compact ---
arcpy.env.workspace = FDS_PATH
all_nds = arcpy.ListDatasets("*", "Network") or []
arcpy.env.workspace = ""

for nd_name in all_nds:
    nd_p = os.path.join(FDS_PATH, nd_name)
    try:
        arcpy.management.Delete(nd_p)
        print(f"Deleted: {nd_name}")
    except Exception as e:
        if arcpy.Exists(nd_p):
            raise RuntimeError(
                f"Cannot delete {nd_name} — ArcGIS Pro has a schema lock.\n"
                f"Remove it from the map (right-click → Remove), then rerun."
            ) from e

arcpy.management.Compact(GDB_PATH)
print("GDB compacted.\n")

# --- Create and build each ND ---
for m in ND_MODES:
    nd_path  = os.path.join(FDS_PATH, m["nd_name"])
    template = m["template"]
    mode     = m["mode"]

    if not os.path.exists(template):
        print(f"[{mode}] SKIPPED — template not found: {os.path.basename(template)}")
        print(f"  Run Step 11a first.\n")
        continue

    # Patch stale source name in-memory
    with open(template, "r", encoding="utf-8") as f:
        xml = f.read()
    if STALE_SOURCE in xml:
        xml = xml.replace(STALE_SOURCE, m["fc_name"])
        print(f"[{mode}] Patched: {STALE_SOURCE!r} → {m['fc_name']!r}")

    tmp_xml = os.path.join(tempfile.gettempdir(), f"{m['nd_name']}_patched.xml")
    with open(tmp_xml, "w", encoding="utf-8") as f:
        f.write(xml)

    # Remove orphan junction FCs
    arcpy.env.workspace = FDS_PATH
    for orphan in (arcpy.ListFeatureClasses(m["nd_name"] + "*") or []):
        arcpy.management.Delete(os.path.join(FDS_PATH, orphan))
    arcpy.env.workspace = ""

    print(f"[{mode}] Creating from template...")
    arcpy.na.CreateNetworkDatasetFromTemplate(tmp_xml, FDS_PATH)

    print(f"[{mode}] Building network...")
    arcpy.na.BuildNetwork(nd_path)
    print(f"[{mode}] Done → {nd_path}\n")

Pre-flight: all source FCs and time fields present.

GDB compacted.

[Walk] Patched: 'Overture_Walk_Segments' → 'Overture_Walk_Routable'
[Walk] Creating from template...
[Walk] Building network...
[Walk] Done → F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Walk_ND

[Bike] Patched: 'Overture_Walk_Segments' → 'Overture_Bike_Routable'
[Bike] Creating from template...
[Bike] Building network...
[Bike] Done → F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Bike_ND

[Drive] Patched: 'Overture_Walk_Segments' → 'Overture_Drive_Routable'
[Drive] Creating from template...
[Drive] Building network...
[Drive] Done → F:\GIS\PROJECTS\Transportation\Streets Network\Streets_Network.gdb\OvertureStreets\Overture_Drive_ND



### Step 11b: Create and Build Network Datasets

---
## Summary

In [30]:
import arcpy, os

print("=" * 62)
print("PIPELINE SUMMARY")
print("=" * 62)

print("\nRaw FCs:")
for fc_name in ["Overture_Segments", "Overture_Connectors", "Overture_Edges_Routable"]:
    path = os.path.join(FDS_PATH, fc_name)
    n    = int(arcpy.management.GetCount(path)[0]) if arcpy.Exists(path) else None
    print(f"  {fc_name:<26} {n:>8,}" if n is not None else f"  {fc_name:<26} NOT FOUND")

print(f"\n{'Mode FC':<30} {'Total':>8} {'In Basin':>10} {'Outside':>10}")
print("-" * 62)

for mode, fc_name, extra in [
    ("Walk",  "Overture_Walk_Routable",  []),
    ("Bike",  "Overture_Bike_Routable",  ["Wilderness_Closed"]),
    ("Drive", "Overture_Drive_Routable", []),
]:
    path = os.path.join(FDS_PATH, fc_name)
    if not arcpy.Exists(path):
        print(f"  {fc_name:<30} NOT FOUND"); continue
    all_f = {f.name for f in arcpy.ListFields(path)}
    total = int(arcpy.management.GetCount(path)[0])
    if "In_Basin" in all_f:
        n_in = sum(1 for r in arcpy.da.SearchCursor(path, ["In_Basin"]) if r[0] == 1)
        print(f"  {fc_name:<30} {total:>8,} {n_in:>10,} {total - n_in:>10,}")
    else:
        print(f"  {fc_name:<30} {total:>8,}")
    if "Wilderness_Closed" in all_f and "Wilderness_Closed" in extra:
        n_w = sum(1 for r in arcpy.da.SearchCursor(path, ["Wilderness_Closed"]) if r[0] == 1)
        print(f"    Wilderness_Closed=1 (bike prohibited): {n_w:,}")
        print(f"    Bike ND query: In_Basin = 1 AND Wilderness_Closed = 0")

print("\nNetwork Datasets:")
for nd_name in ["Overture_Walk_ND", "Overture_Bike_ND", "Overture_Drive_ND"]:
    nd_path = os.path.join(FDS_PATH, nd_name)
    status  = "built" if arcpy.Exists(nd_path) else "not built (template missing — see Step 11)"
    print(f"  {nd_name:<28} {status}")

PIPELINE SUMMARY

Raw FCs:
  Overture_Segments            74,867
  Overture_Connectors         109,079
  Overture_Edges_Routable     147,359

Mode FC                           Total   In Basin    Outside
--------------------------------------------------------------
  Overture_Walk_Routable          145,810     24,948    120,862
  Overture_Bike_Routable          145,590     24,865    120,725
    Wilderness_Closed=1 (bike prohibited): 295
    Bike ND query: In_Basin = 1 AND Wilderness_Closed = 0
  Overture_Drive_Routable          95,285     18,285     77,000

Network Datasets:
  Overture_Walk_ND             built
  Overture_Bike_ND             built
  Overture_Drive_ND            built
